In [ ]:
import json
from pathlib import Path
import optuna

REPO_ROOT = Path('')
HEURISTIC_NB = REPO_ROOT / 'Code' / 'Heuristic.ipynb'

nb_data = json.loads(HEURISTIC_NB.read_text(encoding='utf-8'))
for cell in nb_data['cells']:
    if cell['cell_type'] != 'code':
        continue
    
    src = ''.join(cell['source'])
    
    if 'def executar_heuristica(' in src:
        src = src.split('\nbest_valor_da_solucao,')[0]
   
    exec(src, globals())

print('Funções de Heuristic.ipynb carregadas com sucesso!')

In [ ]:
INSTANCIAS_CALIBRACAO = [
    ('V7', '18h'),
    ('V8', '18h'),
    ('V11', '18h'),
    ('V15', '18h'),
    ('V16', '18h'),
]

TEMPO_POR_INSTANCIA_S = 30.0
N_TRIALS = 30

CACHE_INSTANCIAS = {}
for versao, faixa in INSTANCIAS_CALIBRACAO:
    CACHE_INSTANCIAS[(versao, faixa)] = {
        **carregar_dados_instancia(versao),
        'matriz': carregar_matriz(versao, faixa),
    }

print(f'{len(CACHE_INSTANCIAS)} instâncias carregadas!')


In [ ]:
def avaliar_configuracao(alfa_ini, incremento_alfa, max_it_sem_melhora_alfa, max_iteracoes):
    global df_cars, df_occurrences, df_times, df_availability, df, matriz
    
    custos = []
    for (versao, faixa), dados in CACHE_INSTANCIAS.items():
        df_cars = dados['df_cars']
        df_occurrences = dados['df_occurrences']
        df_times = dados['df_times']
        df_availability = dados['df_availability']
        df = dados['df']
        matriz = dados['matriz']

        makespan, _, _ = executar_heuristica(
            alfa_ini=alfa_ini,
            incremento_alfa=incremento_alfa,
            max_it_sem_melhora_alfa=max_it_sem_melhora_alfa,
            max_iteracoes=max_iteracoes,
            alfa_max=0.5,
            limite_tempo_s=TEMPO_POR_INSTANCIA_S,
            usar_busca_local=True,
            imprimir_infos=False,
        )
        custos.append(makespan)

    return float(np.mean(custos))


def objetivo(trial):
    alfa_ini = trial.suggest_float('alfa_ini', 0.1, 0.5, step=0.05)
    incremento_alfa = trial.suggest_float('incremento_alfa', 0.05, 0.3, step=0.05)
    max_iteracoes = trial.suggest_int('max_iteracoes', 50, 400, step=50)
    max_it_sem_melhora_alfa = trial.suggest_int('max_it_sem_melhora_alfa', 5, 40, step=5)

    return avaliar_configuracao(
        alfa_ini=alfa_ini,
        incremento_alfa=incremento_alfa,
        max_it_sem_melhora_alfa=max_it_sem_melhora_alfa,
        max_iteracoes=max_iteracoes,
    )


sampler = optuna.samplers.TPESampler(seed=42)
estudo = optuna.create_study(direction='minimize', sampler=sampler)
estudo.optimize(objetivo, n_trials=N_TRIALS, show_progress_bar=True)

print('\n================================')
print(f'Melhor makespan médio: {estudo.best_value:.2f}')
print('Melhores parâmetros encontrados:')
for k, v in estudo.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
df_trials = estudo.trials_dataframe().sort_values('value')

df_trials[['number', 'value', 'params_alfa_ini', 'params_incremento_alfa', 'params_max_it_sem_melhora_alfa', 'params_max_iteracoes']].head(10)